# Variables metadata and mapping


This notebook takes in the cmor tables, as well as the ICON mapping and semi-automatically generates the pyku variable configuration. The result of this output must be included by hand and reviewed.

In [ ]:
import json
from itertools import islice
import pandas as pd
import yaml

## CMOR variables and metadata

In [ ]:
url = "https://raw.githubusercontent.com/WCRP-CORDEX/data-request-table/refs/heads/main/cmor-table/datasets.csv"
df = pd.read_csv(url)

cmor_variables = {}

def _to_iso_units(unit):
    if unit == 'K':
        return 'kelvin'
    if unit == 's':
        return 's'
    elif unit == 'degC':
        return 'celsius'
    elif unit == '%':
        return 'percent'
    elif unit == 'J kg-1':
        return 'J kg^-1'
    elif unit == 'kg m-2':
        return 'kg m^-2'
    elif unit == 'kg m-3':
        return 'kg m^-3'
    elif unit == 'kg m-2 s-1':
        return 'kg m^-2 s^-1'
    elif unit == 'W m-2':
        return 'W m^-2'
    elif unit == 'm s-1':
        return 'm s^-1'
    elif unit == 'Pa':
        return 'Pa'
    elif unit == '1':
        return 'dimensionless'
    elif unit == 'm of water equivalent':
        return 'm'
    elif unit == 'm2':
        return 'm^2'
    elif unit == 'm3':
        return 'm^3'
    elif unit == 'm':
        return 'm'
    elif unit == 'm-1':
        return 'm^-1'
    elif unit == 'm-3':
        return 'm^-3'
    elif unit == 'm2 s-1':
        return 'm^2 s^-1'
    elif unit == 'N m-2':
        return 'N m^2'
    elif unit == 'degree':
        return 'degree'
    elif unit == '0.001':
        return '0.001'
    else:
        print(f'unit {unit} not validated')
        return unit
    
for _, row in df.iterrows():
    var_name = row['out_name']
    cmor_variables[var_name] = {
        'long_name': row['long_name'],
        'standard_name': row['standard_name'],
        'units': _to_iso_units(row['units']),
        'cmor_units': row['units'],
        'valid_bounds': [float(-1E20), float(1E20)] # Default
    }

# Print the first three entries
# -----------------------------

first_three_entries = dict(islice(cmor_variables.items(), 3))
print(json.dumps(first_three_entries, indent=4))

with open('./junk/cmor_variables.yaml', 'w') as file:
    yaml.dump(cmor_variables, file, sort_keys=False, default_flow_style=None)

## ICON mapping to CMOR variables

In [ ]:
import re
import yaml
import requests

url = "https://gitlab.dkrz.de/udag/CORDEX-CMIP6_DaSt/-/raw/master/cmor/scripts/tables/ICON-CLM_EUR-12_CORDEX-CMIP6_mapping.txt"

response = requests.get(url)
lines = response.text.splitlines()

icon_variables = {}
pattern = re.compile(r'(\w+)=("[^"]*"|\S+)')

for line in lines:
    line = line.strip()
    if line.startswith('&parameter'):
        matches = pattern.findall(line)
        entry = {k: v.strip('"').strip('/') for k, v in matches}
        cmor_name = entry.get('cmor_name')
        name = entry.get('name')
        level = entry.get('z_axis')

        if name:
            icon_variables[cmor_name] = {
                'name': name,
                'level': level,
            }
            
first_entries = dict(islice(icon_variables.items(), 8))
print(json.dumps(first_entries, indent=4))

with open('./junk/mapping_icon.yaml', 'w') as file:
    yaml.dump(icon_variables, file, sort_keys=False, default_flow_style=False)